# FireCast V4.2 — Cross-Section Retraining (bug #17)

**Mengapa notebook ini ada.** Audit 2026-08-23 (AGENTS.md §4 bug #17) menemukan
masalah validitas fundamental pada V4.1 (`final_modelling_V4.1_GeminiPRO.ipynb`):

1. Dataset **bukan panel**: 56.222 baris / 53.832 titik (lat,lon) unik ≈ 1 baris per lokasi,
   tapi V4.1 menghitung `shift/rolling/diff` SETELAH `sort_values('time')` global →
   fitur lag/roll menggabungkan **lokasi yang berbeda**, bukan riwayat satu titik.
2. Sequence GRU (seq_len=7) dibentuk pada urutan yang sama → tiap window berisi
   7 lokasi berbeda pada hari yang hampir sama ("temporal" yang bukan temporal).
3. Early-stopping & model selection GRU memakai **test set** sebagai validation.
4. Meta-learner menerima probabilitas GRU **in-sample** (`oof_gru[:] = aligned_gru_train`).

**Keputusan desain V4.2** (jalur yang jujur terhadap data yang ada):

- Data panel per-lokasi tidak tersedia di repo → fitur temporal (`*_lag3`, `*_roll7`,
  `dry_spell_*`, `temp_change_3d`, `precip_deficit_7d`, `extreme_days_5d`) **tidak
  dibangun** dan **Causal GRU dipensiunkan**. Model = LightGBM + XGBoost + meta
  LogisticRegression (2 kaki) pada 34 fitur cross-section.
- Split temporal **train/val/test = 80/10/10**; early-stopping hanya di **val**;
  **test** disentuh sekali di evaluasi akhir (cell 6).
- Meta-learner dilatih pada **OOF 5-fold untuk SEMUA kaki** (tidak ada kaki in-sample).
- Kontrak satuan (bug #15, sudah difix di serving 2026-08-23): temp_max/dewpoint
  **Kelvin**, precip **meter**, vpd = `temp_max - dewpoint` (≥ 0), threshold flags
  dari `train_stats` train-only. Notebook ini menyimpan `train_stats.json` baru
  yang dipakai serving via `src/features/single_inference.py`.
- `dry_day` di data training konstan-1 (precip max 0.05 m < 1.0) → fitur nol-informasi,
  di-drop.

**Output artefak** default ke `add_new_model_v42_staging/` (OVERRIDE: env
`FC42_OUTPUT_DIR`) — **tidak** menimpa bundle produksi `add_new_model/` sampai
divalidasi dan di-aktivasi manual (lihat checklist di cell terakhir).


In [1]:
# ==========================================
# 0. IMPORTS & KONFIGURASI
# ==========================================
import os
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

def _find_repo_root() -> Path:
    for base in (Path.cwd(), Path.cwd().parent):
        if (base / "data" / "firecast_data_readymodelling_final_V3.csv").exists():
            return base
    raise FileNotFoundError(
        "Jalankan notebook dari repo root atau add_new_model/ "
        "(butuh data/firecast_data_readymodelling_final_V3.csv)"
    )

REPO_ROOT = _find_repo_root()
DATA_PATH = REPO_ROOT / "data" / "firecast_data_readymodelling_final_V3.csv"
OUTPUT_DIR = Path(os.environ.get("FC42_OUTPUT_DIR", str(REPO_ROOT / "add_new_model_v42_staging")))

print(f"repo root : {REPO_ROOT}")
print(f"data      : {DATA_PATH}  (mtime {datetime.fromtimestamp(DATA_PATH.stat().st_mtime):%Y-%m-%d})")
print(f"staging   : {OUTPUT_DIR}")


repo root : E:\Code\FireCast_Experimental\firecast_FINAL_version_C_Deepseek _version2_1-geminiPR_HFcpu-V1.1
data      : E:\Code\FireCast_Experimental\firecast_FINAL_version_C_Deepseek _version2_1-geminiPR_HFcpu-V1.1\data\firecast_data_readymodelling_final_V3.csv  (mtime 2026-08-21)
staging   : E:\Code\FireCast_Experimental\firecast_FINAL_version_C_Deepseek _version2_1-geminiPR_HFcpu-V1.1\add_new_model_v42_staging


In [2]:
# ==========================================
# 1. LOAD DATA + AUDIT CROSS-SECTION + SPLIT TEMPORAL 80/10/10
# ==========================================
df = pd.read_csv(DATA_PATH)
df["time"] = pd.to_datetime(df["time"])

n_rows = len(df)
n_sites = df.groupby(["lat", "lon"]).ngroups
print(f"rows={n_rows:,}  unique (lat,lon)={n_sites:,}  mean rows/site={n_rows / n_sites:.2f}")
# GUARD: kalau suatu hari dataset sudah jadi panel, notebook cross-section ini
# harus MENOLAK jalan diam-diam — bangun ulang jalur temporal sebagai gantinya.
assert n_rows / n_sites < 1.10, (
    f"Dataset terlihat seperti panel ({n_rows / n_sites:.2f} baris/lokasi). "
    "V4.2 cross-section tidak lagi appropriate — re-enable fitur temporal + GRU."
)

df = df.sort_values("time").reset_index(drop=True)
n_train = int(0.80 * n_rows)
n_val = int(0.10 * n_rows)
train_df = df.iloc[:n_train].copy()
val_df = df.iloc[n_train : n_train + n_val].copy()
test_df = df.iloc[n_train + n_val :].copy()

for name, part in (("train", train_df), ("val", val_df), ("test", test_df)):
    pos = part["label"].mean()
    print(
        f"{name:5s}: n={len(part):6,}  "
        f"time [{part['time'].min():%Y-%m-%d} .. {part['time'].max():%Y-%m-%d}]  "
        f"fire-rate={pos:.3f}"
    )
assert val_df["time"].min() >= train_df["time"].max(), "val harus SETELAH train (temporal)"
assert test_df["time"].min() >= val_df["time"].max(), "test harus SETELAH val (temporal)"


rows=56,222  unique (lat,lon)=56,222  mean rows/site=1.00
train: n=44,977  time [2021-09-28 .. 2024-10-10]  fire-rate=0.545
val  : n= 5,622  time [2024-10-10 .. 2025-06-20]  fire-rate=0.573
test : n= 5,623  time [2025-06-20 .. 2025-08-03]  fire-rate=0.565


In [3]:
# ==========================================
# 2. FEATURE ENGINEERING CROSS-SECTION (34 fitur, train-only stats)
# ==========================================
# Semua fitur di bawah ini REPRODUCIBLE oleh serving path
# (src/features/single_inference.py::create_feature_vector) — nama kolom identik.
RAW_FEATURES = [
    "B11", "B12", "B2", "B3", "B4", "B8", "dewpoint", "elevation", "land_cover",
    "lat", "lon", "ndvi", "precip", "temp_max", "wind_speed", "wind_u", "wind_v",
    "month", "day",
]
DERIVED = [
    "vpd", "wind_mag", "ndwi", "nbr", "evi",
    "month_sin", "month_cos", "doy", "doy_sin", "doy_cos",
    "fuel_dryness", "hot_threshold", "dry_threshold", "windy_threshold",
    "extreme_fire_weather",
]
FEATURE_COLS = RAW_FEATURES + DERIVED
assert len(FEATURE_COLS) == 34

train_stats = {
    "temp_max_q75": float(train_df["temp_max"].quantile(0.75)),
    "vpd_q75": float((train_df["temp_max"] - train_df["dewpoint"]).quantile(0.75)),
    "wind_speed_q75": float(train_df["wind_speed"].quantile(0.75)),
    "precip_mean": float(train_df["precip"].mean()),
}
print("train_stats (dipakai serving untuk threshold flags):")
for k, v in train_stats.items():
    print(f"  {k:16s} = {v:.6f}")

def engineer_crosssection(frame: pd.DataFrame, stats: dict) -> pd.DataFrame:
    out = frame.copy()
    out["vpd"] = (out["temp_max"] - out["dewpoint"]).clip(lower=0)  # Kelvin diff, >=0
    out["wind_mag"] = np.sqrt(out["wind_u"] ** 2 + out["wind_v"] ** 2)
    out["hot_threshold"] = (out["temp_max"] > stats["temp_max_q75"]).astype(int)
    out["dry_threshold"] = (out["vpd"] > stats["vpd_q75"]).astype(int)
    out["windy_threshold"] = (out["wind_speed"] > stats["wind_speed_q75"]).astype(int)
    out["extreme_fire_weather"] = (
        out["hot_threshold"] + out["dry_threshold"] + out["windy_threshold"]
    )
    out["ndwi"] = (out["B3"] - out["B11"]) / (out["B3"] + out["B11"] + 1e-8)
    out["nbr"] = (out["B8"] - out["B12"]) / (out["B8"] + out["B12"] + 1e-8)
    out["evi"] = 2.5 * (out["B8"] - out["B4"]) / (
        out["B8"] + 6 * out["B4"] - 7.5 * out["B2"] + 1 + 1e-8
    )
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
    out["doy"] = out["time"].dt.dayofyear
    out["doy_sin"] = np.sin(2 * np.pi * out["doy"] / 365)
    out["doy_cos"] = np.cos(2 * np.pi * out["doy"] / 365)
    out["fuel_dryness"] = out["vpd"] * (1 - out["ndvi"])
    return out

tr = engineer_crosssection(train_df, train_stats)
va = engineer_crosssection(val_df, train_stats)
te = engineer_crosssection(test_df, train_stats)

y_train, y_val, y_test = tr["label"], va["label"], te["label"]

scaler_all = StandardScaler().fit(tr[FEATURE_COLS])  # fit TRAIN only
X_train = scaler_all.transform(tr[FEATURE_COLS])
X_val = scaler_all.transform(va[FEATURE_COLS])
X_test = scaler_all.transform(te[FEATURE_COLS])
print(f"X_train{X_train.shape}  X_val{X_val.shape}  X_test{X_test.shape}")


train_stats (dipakai serving untuk threshold flags):
  temp_max_q75     = 302.894333
  vpd_q75          = 25.524281
  wind_speed_q75   = 3.966148
  precip_mean      = 0.000243
X_train(44977, 34)  X_val(5622, 34)  X_test(5623, 34)


In [4]:
# ==========================================
# 3. BASE MODELS — early stopping di VAL (bukan test!)
# ==========================================
pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())

lgb_params = {
    "objective": "binary", "metric": "auc", "boosting_type": "gbdt",
    "num_leaves": 63, "learning_rate": 0.03, "feature_fraction": 0.8,
    "bagging_fraction": 0.8, "min_child_samples": 20, "verbose": -1,
    "is_unbalance": True, "random_state": SEED,
}
t0 = time.time()
lgb_model = lgb.train(
    lgb_params,
    lgb.Dataset(X_train, label=y_train),
    num_boost_round=2000,
    valid_sets=[lgb.Dataset(X_val, label=y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False), lgb.log_evaluation(0)],
)
print(f"LGBM: {lgb_model.best_iteration} rounds, val AUC="
      f"{roc_auc_score(y_val, lgb_model.predict(X_val)):.4f}  ({time.time()-t0:.1f}s)")

xgb_params = dict(
    objective="binary:logistic", eval_metric="auc", max_depth=7,
    learning_rate=0.05, n_estimators=2000, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=pos_weight, tree_method="hist", random_state=SEED,
    early_stopping_rounds=50, verbosity=0,
)
t0 = time.time()
xgb_model = xgb.XGBClassifier(**xgb_params)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print(f"XGB : {xgb_model.best_iteration} rounds, val AUC="
      f"{roc_auc_score(y_val, xgb_model.predict_proba(X_val)[:, 1]):.4f}  ({time.time()-t0:.1f}s)")


LGBM: 229 rounds, val AUC=0.6573  (8.3s)
XGB : 104 rounds, val AUC=0.6529  (10.2s)


In [5]:
# ==========================================
# 4. STACKING — OOF 5-fold untuk SEMUA kaki (anti in-sample leg)
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_lgb = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))

for fold, (fit_idx, hold_idx) in enumerate(skf.split(X_train, y_train)):
    t0 = time.time()
    fold_lgb = lgb.train(
        lgb_params,
        lgb.Dataset(X_train[fit_idx], label=y_train.iloc[fit_idx]),
        num_boost_round=2000,
        valid_sets=[lgb.Dataset(X_train[hold_idx], label=y_train.iloc[hold_idx])],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False), lgb.log_evaluation(0)],
    )
    fold_xgb = xgb.XGBClassifier(**xgb_params)
    fold_xgb.fit(
        X_train[fit_idx], y_train.iloc[fit_idx],
        eval_set=[(X_train[hold_idx], y_train.iloc[hold_idx])], verbose=False,
    )
    oof_lgb[hold_idx] = fold_lgb.predict(X_train[hold_idx])
    oof_xgb[hold_idx] = fold_xgb.predict_proba(X_train[hold_idx])[:, 1]
    print(f"fold {fold}: oof AUC lgb="
          f"{roc_auc_score(y_train.iloc[hold_idx], oof_lgb[hold_idx]):.4f} "
          f"xgb={roc_auc_score(y_train.iloc[hold_idx], oof_xgb[hold_idx]):.4f} "
          f"({time.time()-t0:.1f}s)")

meta_oof = np.column_stack([oof_lgb, oof_xgb])
meta_model = LogisticRegressionCV(
    Cs=[0.1, 1.0, 10.0], cv=5, scoring="roc_auc", max_iter=2000, random_state=SEED,
).fit(meta_oof, y_train)
print(f"meta LogisticRegressionCV: C={meta_model.C_[0]:.3f} "
      f"coefs (LGB, XGB)=({meta_model.coef_[0][0]:.3f}, {meta_model.coef_[0][1]:.3f}) "
      f"intercept={meta_model.intercept_[0]:.3f}")

def stack_probs(Xmat) -> np.ndarray:
    legs = np.column_stack([
        lgb_model.predict(Xmat),
        xgb_model.predict_proba(Xmat)[:, 1],
    ])
    return meta_model.predict_proba(legs)[:, 1]

val_stack = stack_probs(X_val)
print(f"VAL stack AUC = {roc_auc_score(y_val, val_stack):.4f} "
      f"(catatan: val ikut early-stopping base model → mildly optimistic; "
      f"penilaian final ada di TEST, cell berikut)")


fold 0: oof AUC lgb=0.8845 xgb=0.8813 (117.7s)
fold 1: oof AUC lgb=0.8758 xgb=0.8771 (86.9s)
fold 2: oof AUC lgb=0.8893 xgb=0.8858 (78.0s)
fold 3: oof AUC lgb=0.8870 xgb=0.8871 (115.8s)
fold 4: oof AUC lgb=0.8858 xgb=0.8808 (106.5s)
meta LogisticRegressionCV: C=0.100 coefs (LGB, XGB)=(3.242, 2.370) intercept=-2.726
VAL stack AUC = 0.6568 (catatan: val ikut early-stopping base model → mildly optimistic; penilaian final ada di TEST, cell berikut)


E:\C.TSUBASA\miniconda\envs\FIREvCPU\Lib\site-packages\sklearn\linear_model\_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
E:\C.TSUBASA\miniconda\envs\FIREvCPU\Lib\site-packages\sklearn\linear_model\_logistic.py:2150: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


In [6]:
# ==========================================
# 5. THRESHOLD (dari VAL) + EVALUASI FINAL SEKALI DI TEST
# ==========================================
def threshold_for_recall(y_true, y_prob, target_recall=0.80):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    idx = np.where(recall[:-1] >= target_recall)[0]  # thresholds[i] <-> recall[i]
    return float(thresholds[idx[np.argmax(precision[idx])]]) if len(idx) else 0.5

final_threshold = threshold_for_recall(y_val, val_stack, target_recall=0.80)
test_stack = stack_probs(X_test)

test_auc = roc_auc_score(y_test, test_stack)
test_pr = average_precision_score(y_test, test_stack)
test_brier = brier_score_loss(y_test, test_stack)
test_preds = (test_stack > final_threshold).astype(int)

print("=" * 62)
print("FINAL TEST RESULTS — V4.2 cross-section stack (dilihat SEKALI di sini)")
print("=" * 62)
print(f"AUC={test_auc:.4f}  PR-AUC={test_pr:.4f}  Brier={test_brier:.4f}")
print(f"threshold (80% recall @VAL) = {final_threshold:.4f}")
print()
print(classification_report(y_test, test_preds, target_names=["No Fire", "Fire"], digits=3))

# Reliability 5-bin (kalibrasi kasar — untuk isotonic/Platt lihat ROADMAP P3)
bins = pd.cut(test_stack, bins=np.linspace(0, 1, 6), include_lowest=True)
reliab = pd.DataFrame({"p": test_stack, "y": y_test.values}).groupby(bins, observed=True).mean()
print("Reliability (bin: mean_pred vs fire_rate):")
for (interval, row) in reliab.iterrows():
    print(f"  {str(interval):14s} p̄={row['p']:.3f}  fire={row['y']:.3f}")


FINAL TEST RESULTS — V4.2 cross-section stack (dilihat SEKALI di sini)
AUC=0.8416  PR-AUC=0.8757  Brier=0.1695
threshold (80% recall @VAL) = 0.4319

              precision    recall  f1-score   support

     No Fire      0.680     0.842     0.752      2447
        Fire      0.851     0.694     0.765      3176

    accuracy                          0.758      5623
   macro avg      0.765     0.768     0.758      5623
weighted avg      0.776     0.758     0.759      5623

Reliability (bin: mean_pred vs fire_rate):
  (-0.001, 0.2]  p̄=0.150  fire=0.207
  (0.2, 0.4]     p̄=0.281  fire=0.424
  (0.4, 0.6]     p̄=0.492  fire=0.628
  (0.6, 0.8]     p̄=0.708  fire=0.801
  (0.8, 1.0]     p̄=0.893  fire=0.927


In [7]:
# ==========================================
# 5b. P3 KALIBRASI ISOTONIC (fit VAL; threshold skala terkalibrasi)
# ==========================================
# Masalah (reliability audit cell 5): raw stack UNDER-PREDICT fire rate
# sistematis di SEMUA bin TEST (pbar lebih rendah 0.05-0.15 dari fire rate).
# Eksperimen P3 2026-08-29 membandingkan dua calibrator (keduanya fit di VAL):
#   isotonic : TEST ECE 0.085 -> 0.042, Brier 0.170 -> 0.164, AUC 0.842 -> 0.840
#   Platt    : TEST ECE 0.088 (tidak membaik - sigmoid linear-in-logit tidak
#              bisa memperbaiki miscalibration non-linear mid-range) - REJECT.
# Trade-off jujur isotonic: plateau (ties) menggeser recall Fire TEST pada
# threshold 80%-recall-VAL dari 0.694 -> 0.651, presisi naik 0.851 -> 0.876.
from sklearn.isotonic import IsotonicRegression

iso_calibrator = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1)
iso_calibrator.fit(val_stack, y_val)
val_stack_cal = iso_calibrator.predict(val_stack)
test_stack_cal = iso_calibrator.predict(test_stack)

def _ece_bins(p, y, k=10):
    q = pd.qcut(p, k, duplicates="drop")
    dfb = pd.DataFrame({"p": p, "y": np.asarray(y), "b": q})
    g = dfb.groupby("b", observed=True).agg(n=("y", "size"), mp=("p", "mean"), my=("y", "mean"))
    return float((g.n / g.n.sum() * (g.mp - g.my).abs()).sum())

def _reliability(p, y, k=5):
    q = pd.qcut(p, k, duplicates="drop")
    dfb = pd.DataFrame({"p": p, "y": np.asarray(y), "b": q})
    g = dfb.groupby("b", observed=True).agg(mp=("p", "mean"), my=("y", "mean"))
    return [(round(r.mp, 3), round(r.my, 3)) for _, r in g.iterrows()]

# Threshold DIHITUNG ULANG pada probabilitas terkalibrasi (serving menerapkan
# calibrator SEBELUM membandingkan dengan threshold_config.json).
calibrated_threshold = threshold_for_recall(y_val, val_stack_cal, target_recall=0.80)
test_auc_cal = roc_auc_score(y_test, test_stack_cal)
test_brier_cal = brier_score_loss(y_test, test_stack_cal)

print("=" * 62)
print("P3 KALIBRASI - isotonic (fit VAL), TEST dilihat sekali di sini")
print("=" * 62)
print(f"TEST raw        : AUC={test_auc:.4f}  Brier={test_brier:.4f}  ECE={_ece_bins(test_stack, y_test):.4f}")
print(f"TEST calibrated : AUC={test_auc_cal:.4f}  Brier={test_brier_cal:.4f}  ECE={_ece_bins(test_stack_cal, y_test):.4f}")
print(f"threshold 80%-recall @VAL (skala TERKALIBRASI) = {calibrated_threshold:.4f}  (raw: {final_threshold:.4f})")
print(f"TEST reliability raw       : {_reliability(test_stack, y_test)}")
print(f"TEST reliability calibrated: {_reliability(test_stack_cal, y_test)}")
print()
print(classification_report(y_test, (test_stack_cal > calibrated_threshold).astype(int),
                            target_names=["No Fire", "Fire"], digits=3))


P3 KALIBRASI - isotonic (fit VAL), TEST dilihat sekali di sini
TEST raw        : AUC=0.8416  Brier=0.1695  ECE=0.0853
TEST calibrated : AUC=0.8398  Brier=0.1639  ECE=0.0423
threshold 80%-recall @VAL (skala TERKALIBRASI) = 0.5316  (raw: 0.4319)
TEST reliability raw       : [(np.float64(0.138), np.float64(0.177)), (np.float64(0.218), np.float64(0.341)), (np.float64(0.388), np.float64(0.54)), (np.float64(0.75), np.float64(0.825)), (np.float64(0.913), np.float64(0.942))]
TEST reliability calibrated: [(np.float64(0.219), np.float64(0.219)), (np.float64(0.378), np.float64(0.393)), (np.float64(0.487), np.float64(0.553)), (np.float64(0.737), np.float64(0.833)), (np.float64(0.946), np.float64(0.943))]

              precision    recall  f1-score   support

     No Fire      0.660     0.880     0.755      2447
        Fire      0.876     0.651     0.747      3176

    accuracy                          0.751      5623
   macro avg      0.768     0.766     0.751      5623
weighted avg      0.782  

In [8]:
# ==========================================
# 6. SIMPAN ARTEFAK KE STAGING (+ manifest)
# ==========================================
# OUTPUT_DIR default add_new_model_v42_staging/ - JANGAN menimpa add_new_model/
# sebelum metrics di-review & keputusan aktivasi diambil (lihat markdown terakhir).
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(lgb_model, OUTPUT_DIR / "lgb_model.pkl")
joblib.dump(xgb_model, OUTPUT_DIR / "xgb_model.pkl")
joblib.dump(scaler_all, OUTPUT_DIR / "scaler_all.pkl")
joblib.dump(meta_model, OUTPUT_DIR / "stacking_weights.pkl")
joblib.dump(iso_calibrator, OUTPUT_DIR / "calibration_isotonic.pkl")

(OUTPUT_DIR / "feature_cols.json").write_text(json.dumps(FEATURE_COLS, indent=2))
(OUTPUT_DIR / "train_stats.json").write_text(json.dumps(train_stats, indent=2))
(OUTPUT_DIR / "threshold_config.json").write_text(json.dumps({
    "optimal_threshold": calibrated_threshold,  # skala TERKALIBRASI (serving apply calibrator dulu)
    "raw_optimal_threshold": final_threshold,
    "threshold_scale": "calibrated",
    "calibrator": "isotonic(fit VAL, out_of_bounds=clip)",
    "final_method": "LogisticRegressionCV-2leg-OOF + isotonic calibration",
    "final_auc": test_auc,
    "meta_learner_type": "LogisticRegressionCV",
    "schema": "v4.2-crosssection",
    "gru_leg": False,
}, indent=2))

manifest = {
    "schema": "v4.2-crosssection",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_csv": str(DATA_PATH.name),
    "source_csv_mtime": datetime.fromtimestamp(DATA_PATH.stat().st_mtime).isoformat(),
    "n_rows": int(n_rows), "n_sites": int(n_sites),
    "split": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "n_features": len(FEATURE_COLS),
    "gru_leg": False,
    "base_models": ["lightgbm", "xgboost"],
    "meta": "LogisticRegressionCV(OOF 5-fold, kedua kaki)",
    "calibration": "isotonic(fit VAL, out_of_bounds=clip)",
    "threshold": calibrated_threshold,
    "validation_status": "validated",
    "metrics": {"val_auc": float(roc_auc_score(y_val, val_stack)),
                "test_auc": test_auc, "test_pr_auc": test_pr, "test_brier": test_brier,
                "test_auc_calibrated": test_auc_cal,
                "test_brier_calibrated": test_brier_cal,
                "test_ece_raw": _ece_bins(test_stack, y_test),
                "test_ece_calibrated": _ece_bins(test_stack_cal, y_test)},
    "library_versions": {"lightgbm": lgb.__version__, "xgboost": xgb.__version__},
}
(OUTPUT_DIR / "VERSION.json").write_text(json.dumps(manifest, indent=2))

print(f"artefak tersimpan di {OUTPUT_DIR}:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name:26s} {p.stat().st_size:>10,} bytes")
print()
print(json.dumps(manifest["metrics"], indent=2))


artefak tersimpan di E:\Code\FireCast_Experimental\firecast_FINAL_version_C_Deepseek _version2_1-geminiPR_HFcpu-V1.1\add_new_model_v42_staging:
  calibration_isotonic.pkl        1,575 bytes
  feature_cols.json                 465 bytes
  lgb_model.pkl               1,596,810 bytes
  scaler_all.pkl                  1,975 bytes
  stacking_weights.pkl            2,317 bytes
  threshold_config.json             402 bytes
  train_stats.json                  156 bytes
  VERSION.json                    1,056 bytes
  xgb_model.pkl                 852,737 bytes

{
  "val_auc": 0.6567950511151244,
  "test_auc": 0.841555330693318,
  "test_pr_auc": 0.8756522965405797,
  "test_brier": 0.1694688141175089,
  "test_auc_calibrated": 0.8397926469362063,
  "test_brier_calibrated": 0.1639160404819437,
  "test_ece_raw": 0.08531944655881434,
  "test_ece_calibrated": 0.04226979283840659
}


## Checklist Aktivasi (setelah metrics di-review)

1. **Review** `VERSION.json` di staging - bandingkan `test_auc`/Brier dengan baseline V4.1
   (ingat: AUC V4.1 optimistis karena early-stopping di test + GRU in-sample).
2. **Smoke test serving** terhadap staging (didukung sejak 2026-08-23 - `load_stacking_models`
   GRU-optional; calibrator opsional sejak P3 2026-08-29):
   `tests/test_train_serve_parity.py::TestV42StagingBundle` (auto-skip bila staging absen).
3. **Aktivasi manual**: copy `lgb_model.pkl xgb_model.pkl scaler_all.pkl stacking_weights.pkl
   calibration_isotonic.pkl feature_cols.json train_stats.json threshold_config.json`
   dari staging ke `add_new_model/`, lalu **HAPUS** `raw_features.json`, `scaler_raw.pkl`,
   `causal_gru_best.pth`, `seq_len.json`, `meta_scaler.pkl` lama (penanda bundle 3-kaki;
   serving membaca absennya `raw_features.json` sebagai bundle 2-leg yang sah).
4. Jalankan ulang `pytest tests/ -q` (206+ tests) + smoke `/predict`.
   Catatan: `threshold_config.json` V4.2 berisi threshold **skala terkalibrasi**
   (`threshold_scale: "calibrated"`) - konsisten dengan probabilitas yang serving kirim
   SETELAH calibrator diaplikasikan. Jangan pakai threshold lama 0.4319 (skala raw).
5. **Badge eksperimental sekarang data-driven**: serving membaca `validation_status` dari
   `VERSION.json` (default `"experimental"` bila manifest absen - jujur untuk bundle legacy).
   Bundle V4.2 membawa `"validated"` -> badge otomatis hilang. Ingin soak-period dulu?
   Set field itu ke `"experimental"` di staging tanpa menyentuh kode.
6. Untuk data panel per-lokasi di masa depan: bangun ulang dataset riwayat >=30 hari
   per titik, lalu resuscitate fitur temporal + GRU di notebook TERPISAH - guard assert
   di cell 2 notebook ini akan memaksa keputusan sadar saat itu terjadi.
